In [1]:
# %pip install -U scikit-learn
# %pip install -U kaleido
# %pip install -U plotly
# !plotly_get_chrome
# %pip install -U aeon
# %pip install -U nbformat jupyter ipywidgets

In [2]:
import pandas as pd
import plotly.express as px
from plotly.io import write_image
from sklearn.manifold import TSNE


In [3]:
# dataset loading
df = pd.read_excel('all-results.xlsx', index_col=0, header=0, sheet_name='Sheet1')
df


,DTW_D,ROCKET,HC2,Hydra,MR+Hydra,DLinear,LightTS,MTS-Mixer,TimesNet,ModernTCN,...,TimeMixer++,FEDformer,ETSformer,Crossformer,PatchTST,GPT4TS,iTransformer,InterpGN,TSCMamba,MambaSL
EC,32.319392,44.486692,52.851711,54.372624,60.076046,31.178707,32.319392,30.798479,33.460076,32.319392,...,34.600760,33.840304,33.079848,43.346008,32.699620,32.319392,31.939163,28.897338,30.798479,42.585551
FD,52.866061,61.606129,61.379115,60.981839,61.293984,69.494892,68.473326,68.530079,70.147560,66.997730,...,69.665153,69.892168,68.331442,69.636776,68.076050,68.615210,68.416572,63.791146,70.204313,69.296254
HW,60.705882,59.294118,56.000000,56.352941,53.529412,23.764706,25.058824,19.647059,37.529412,30.117647,...,33.647059,33.411765,35.647059,30.352941,29.529412,34.117647,31.294118,58.705882,45.058824,60.823529
HB,71.707317,75.609756,78.536585,76.585366,77.073171,76.585366,77.560976,79.024390,83.902439,78.048780,...,80.487805,78.048780,78.536585,79.024390,73.658537,79.024390,78.048780,80.487805,78.536585,80.487805
JV,95.945946,83.783784,98.378378,97.837838,98.378378,96.486486,96.756757,95.945946,98.648649,98.648649,...,99.189189,97.567568,98.378378,98.378378,97.027027,98.108108,98.108108,99.729730,98.378378,98.648649
PEMS,71.098266,84.393064,97.687861,98.265896,82.658960,82.658960,87.861272,82.080925,87.861272,86.705202,...,89.017341,89.017341,87.861272,89.595376,89.017341,88.439306,91.329480,86.127168,90.173410,85.549133
SRS1,77.474403,85.665529,89.078498,86.689420,94.880546,92.832765,92.832765,78.839590,92.150171,93.515358,...,92.150171,75.085324,92.150171,93.856655,89.078498,93.174061,93.856655,90.443686,92.832765,92.491468
SRS2,53.888889,55.555556,54.444444,58.888889,55.000000,57.222222,59.444444,58.333333,60.000000,60.000000,...,60.555556,59.444444,60.000000,61.666667,61.666667,60.555556,58.888889,58.333333,62.222222,65.000000
SAD,97.226012,99.226921,99.135971,99.135971,99.045020,96.953161,98.226467,99.454297,99.545248,99.499773,...,99.545248,99.454297,98.999545,98.863120,98.817644,99.590723,99.317872,99.863574,95.816280,99.954525
UW,90.312500,94.062500,94.062500,94.375000,94.062500,82.812500,89.062500,83.125000,88.437500,88.437500,...,90.625000,79.687500,89.687500,88.437500,89.375000,89.062500,90.000000,88.437500,95.312500,93.437500


### t-SNE

In [4]:
for i, p in enumerate(range(3, 11, 2)):
    # t-SNE fitting
    tsne = TSNE(n_components=2, perplexity=p, metric="correlation",
                max_iter=int(1e6), random_state=0) 
    X_tsne = tsne.fit_transform(df)

    # t-SNE visualization DataFrame
    df_tsne = pd.DataFrame(X_tsne, columns=['x', 'y'])
    df_tsne['dataset'] = df.index
    df_tsne['acc'] = df.mean(axis=1).values

    fig = px.scatter(df_tsne, x='x', y='y', text='dataset', 
                    labels={'x': 'component 1', 'y': 'component 2'},
                    color='acc',
                    color_continuous_scale=px.colors.sequential.Viridis,
                    size='acc',
                    size_max=8,
                    opacity=0.7)
    # fig size
    fig.update_layout(width=800, height=400)
    # title
    fig.update_layout(title=f'Visualization of UEA datasets using t-SNE (perplexity={p}, metric=correlation)', 
                        title_x=0.5,  # center the title
                        title_y=0.98,
                        title_font=dict(size=20))
    # tight layout
    fig.update_layout(
        coloraxis_colorbar=dict(
            title="avg.<br>acc",
            title_font=dict(size=13),
            tickmode="array",
            tickvals=list(range(70,80,2)),
            lenmode="fraction",
            thickness=20,
            len=1,
            yanchor="middle",
            y=0.5,
            xanchor="left",
            x=1.0,
        ),
    )
    fig.update_layout(margin=dict(l=0, r=0, t=30, b=15))
    fig.update_xaxes(title_standoff=7)
    fig.update_yaxes(title_standoff=7)

    # text size
    fig.update_traces(textfont_size=18)
    fig.update_traces(textposition='bottom center')
    fig.update_layout(showlegend=False)
    fig.show()

    # # save the figure
    write_image(fig, f'figures_tmp/tsne_perplexity_{p}_dataset.pdf', width=800, height=400, scale=2, format='pdf')

### UMAP

In [5]:
# %pip install umap-learn

In [6]:
from umap import UMAP

for i, k in enumerate([3,6,9,12]):
    reduction = UMAP(n_components=2, n_neighbors=k, metric="correlation",
                     random_state=0, n_jobs=1)
    X_umap = reduction.fit_transform(df)
    df_umap = pd.DataFrame(X_umap, columns=['x', 'y'])
    df_umap['dataset'] = df.index
    df_umap['acc'] = df.mean(axis=1).values


    fig = px.scatter(df_umap, x='x', y='y', text='dataset', 
                    labels={'x': 'component 1', 'y': 'component 2'},
                    color='acc',
                    color_continuous_scale=px.colors.sequential.Viridis,
                    # size='acc',
                    # size_max=8,
                    opacity=0.7)
    fig.update_traces(marker={'size': 10})
    # fig size
    fig.update_layout(width=800, height=400)
    # title
    fig.update_layout(title=f'Visualization of UEA datasets using UMAP (n_neighbors={k}, metric=correlation)', 
                        title_x=0.5,  # center the title
                        title_y=0.98,
                        title_font=dict(size=20))
    # tight layout
    fig.update_layout(
        coloraxis_colorbar=dict(
            title="avg.<br>acc",
            title_font=dict(size=13),
            tickmode="array",
            tickvals=list(range(0,101,10)),
            lenmode="fraction",
            thickness=20,
            len=1,
            yanchor="middle",
            y=0.5,
            xanchor="left",
            x=1.0,
        ),
    )
    fig.update_layout(margin=dict(l=0, r=0, t=30, b=15))
    fig.update_xaxes(title_standoff=7)
    fig.update_yaxes(title_standoff=7)

    # text size
    fig.update_traces(textfont_size=20)
    fig.update_traces(textposition='bottom center')
    fig.update_layout(showlegend=False)
    fig.show()

    # # save the figure
    write_image(fig, f'figures_tmp/umap_neighbors{k}_dataset.pdf', width=800, height=400, scale=2, format='pdf')